# Interactive Slip Distribution Mapping

This notebook provides an interactive interface to generate and inspect slip distribution maps produced by pyANTI-FASc.

The main numerical workflow should be executed beforehand using the command-line interface.  
This notebook is intended for post-processing and visualization only.

## Notes

This notebook is designed for visualization and post-processing.

For best performance:

- Run the full pyANTI-FASc pipeline from the command line
- Use this notebook only after the output files have been generated
- Avoid re-running map generation unless the input slip files have changed

The first cells can be used to visualize the maps through the geojson files already computed by the code
The other ones might be used to generate interactive html files and visualize them


## 1. Imports and configuration

In this section, we import the required Python modules and load the pyANTI-FASc configuration file.

The plotting functions are imported from the local Python script:

```text
plot_slip_dsitribution.py
```

The configuration file is read from:

```text
../config_files/Parameters/input.json
```

The output will be searched among the results in the folder:

```text
../output
```



In [1]:
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import folium
from branca.colormap import LinearColormap

from utils.plot_slip_distribution import ascii_to_geojson, plot_slip_map, progress, plot_geojson_property, generate_slip_maps

with open("../config_files/Parameters/input.json") as fid:
    Param = json.load(fid)

base_folder = Path("../output")

## 2. Select the output directory

The pyANTI-FASc output directory contains a nested folder structure describing the selected run.

The following widgets allow the user to select:

1. Event
2. Rigidity distribution
3. Magnitude
4. Scaling law

Within the final selected folder, select the slip distribution you want to plot

In [2]:
levels = [
    "event",
    "rigidity distribution",
    "magnitude",
    "scaling law",
]

out = widgets.Output()

folder_dropdowns = []

geojson_dropdown = widgets.Dropdown(
    options=[],
    description="GeoJSON:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="650px")
)

property_dropdown = widgets.Dropdown(
    options=["slip", "rake"],
    value="slip",
    description="Feature:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="350px")
)

tile_options = {
    "CartoDB Positron": {
        "tiles": "CartoDB positron",
        "attr": None
    },
    "CartoDB Dark Matter": {
        "tiles": "CartoDB dark_matter",
        "attr": None
    },
    "CartoDB Voyager": {
        "tiles": "CartoDB Voyager",
        "attr": None
    },
    "CartoDB Positron No Labels": {
        "tiles": "CartoDB positron_nolabels",
        "attr": None
    },
    "OpenTopoMap": {
        "tiles": "https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
        "attr": "Map data: © OpenStreetMap contributors, SRTM | Map style: © OpenTopoMap (CC-BY-SA)"
    }
}

tile_dropdown = widgets.Dropdown(
    options=list(tile_options.keys()),
    value="CartoDB Positron",
    description="Basemap:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="500px")
)


plot_button = widgets.Button(
    description="Plot GeoJSON",
    button_style="success"
)

def list_dirs(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p.name for p in folder.iterdir() if p.is_dir()])

def selected_folder():
    folder = base_folder
    for dd in folder_dropdowns:
        if dd.value is not None:
            folder = folder / dd.value
    return folder

def build_folder_until(index):
    folder = base_folder
    for i in range(index):
        value = folder_dropdowns[i].value
        if value is not None:
            folder = folder / value
    return folder

def update_geojson_dropdown():
    folder = selected_folder()
    geojson_files = sorted(folder.glob("Slip4H*.json"))

    geojson_dropdown.options = [f.name for f in geojson_files]

    if geojson_files:
        geojson_dropdown.value = geojson_files[0].name
    else:
        geojson_dropdown.value = None

def update_folder_dropdowns(change=None):
    for i, dd in enumerate(folder_dropdowns):
        parent = build_folder_until(i)
        choices = list_dirs(parent)

        old_value = dd.value
        dd.options = choices

        if old_value in choices:
            dd.value = old_value
        elif choices:
            dd.value = choices[0]
        else:
            dd.value = None

    update_geojson_dropdown()

for level in levels:
    dd = widgets.Dropdown(
        description=level + ":",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="650px")
    )
    dd.observe(update_folder_dropdowns, names="value")
    folder_dropdowns.append(dd)



def on_plot_clicked(button):
    with out:
        clear_output()

        folder = selected_folder()

        if not geojson_dropdown.value:
            print(f"No GeoJSON files found in: {folder}")
            return

        geojson_file = folder / geojson_dropdown.value
        property_name = property_dropdown.value
        tile_config = tile_options[tile_dropdown.value]

        print(f"Selected folder: {folder}")
        print(f"GeoJSON: {geojson_file.name}")
        print(f"Feature: {property_name}")
        print(f"Basemap: {tile_dropdown.value}")

        m = plot_geojson_property(geojson_file, property_name,tile_config=tile_config)

        if m is not None:
            display(m)

html_button = widgets.Button(
    description="Generate HTML maps",
    button_style="warning",
    layout=widgets.Layout(width="180px")
)

def on_html_clicked(button):
    with out:
        clear_output()

        folder = selected_folder()
        print(f"Selected folder: {folder}")
        print("Generating HTML maps...")

        generate_slip_maps(folder, Param)

        print("Done.")
        update_geojson_dropdown()

html_button.on_click(on_html_clicked)

plot_button.on_click(on_plot_clicked)

update_folder_dropdowns()

display(
    *folder_dropdowns,
    geojson_dropdown,
    property_dropdown,
    tile_dropdown,
    widgets.HBox([plot_button, html_button]),
    out
)

Dropdown(description='event:', layout=Layout(width='650px'), options=('ITCF00G_Hazard_slip_ITC', 'ITCF03F_Haza…

Dropdown(description='rigidity distribution:', layout=Layout(width='650px'), options=('homogeneous_mu', 'varia…

Dropdown(description='magnitude:', layout=Layout(width='650px'), options=('5_5000', '5_6000', '5_7000', '5_800…

Dropdown(description='scaling law:', layout=Layout(width='650px'), options=('WellsCopp1994_Normal',), style=De…

Dropdown(description='GeoJSON:', layout=Layout(width='650px'), options=(), style=DescriptionStyle(description_…

Dropdown(description='Feature:', layout=Layout(width='350px'), options=('slip', 'rake'), style=DescriptionStyl…

Dropdown(description='Basemap:', layout=Layout(width='500px'), options=('CartoDB Positron', 'CartoDB Dark Matt…

Output()

## 3. ⬆️🙄 Generate slip distribution maps 🙄⬆️

After selecting the desired output folder, click one of the button below to generate the slip distribution maps.

For each selection, the notebook will:
1. Plot the selected GeoJSON (visualizing either slip or rake) 
2. Generate an interactive Folium HTML map and save the resulting `.html` file in the same output folder

⬇👇**If you have generated HTML maps, run the next window to visualize them**👇⬇️



## 4. Inspect generated HTML maps

The generated maps are saved as HTML files.

They can be opened directly in a browser, or displayed inside this notebook using an embedded HTML frame.

In [3]:
from pathlib import Path
import shutil
import ipywidgets as widgets
from IPython.display import display, IFrame, clear_output

folder = Path(selected_folder())
html_files = sorted(folder.glob("Slip4H*.html"))

out = widgets.Output()

dropdown = widgets.Dropdown(
    options=[f.name for f in html_files],
    description="Map:",
    layout=widgets.Layout(width="550px")
)

def show_map(change=None):
    with out:
        clear_output()

        if not dropdown.value:
            print("No HTML files found.")
            return

        src_file = folder / dropdown.value
        preview_file = Path("preview_map.html")

        shutil.copy(src_file, preview_file)

        print(f"Showing: {src_file}")
        display(IFrame("preview_map.html", width=950, height=650))

dropdown.observe(show_map, names="value")

display(dropdown, out)

if html_files:
    dropdown.value = html_files[0].name
    show_map()
else:
    print("No HTML files found.")

Dropdown(description='Map:', layout=Layout(width='550px'), options=('Slip4HySea00004_001.html', 'Slip4HySea000…

Output()